<a href="https://colab.research.google.com/github/RohanYashraj/ifoa-workshop/blob/main/notebooks_v2/01_genai_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · GenAI Basics — your first calls to the reasoner

**Agentic AI for Health Actuaries** · IAI Seminar · 25 August 2026 · Hub: `github.com/rohanyashraj/ifoa-workshop`

> All data in this notebook is **hypothetical** — ABC Health is a fictional entity calibrated to plausible Indian health insurance experience, for teaching only.

**Used in:** Session 1, Part 2 (The Reasoner).
**You will:** make your first Gemini API call, practise the CCCE prompt discipline, watch a hallucination happen on demand, and get guaranteed-parseable JSON out of an LLM.

**Setup (2 minutes):**
1. Get a free Gemini API key at [aistudio.google.com](https://aistudio.google.com) → *Get API key*.
2. In Colab, click the **key icon** (left sidebar) → *Add new secret* → name it `GOOGLE_API_KEY`, paste the key, toggle notebook access ON.
3. Run the cells top to bottom (`Runtime → Run all` after setup).

In [3]:
%pip install -q "google-genai==2.19.0" "google-auth==2.49.0"

/Users/mohithsai/Downloads/Kasyap/IAI_25082026_notebooks/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## §1 · Auth — the key never appears in the notebook
Colab Secrets keeps the key out of the notebook file. This is the same hygiene you will use for every agent you ship: secrets live in a store, never in code.

In [4]:
import os
from google import genai
from IPython.display import Markdown, display
from dotenv import load_dotenv
#from google.colab import userdata   # Colab-only; see comment below for local Jupyter

#os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# Local Jupyter alternative:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

client = genai.Client()
MODEL = "gemini-3.5-flash-lite"   # PINNED — silent model drift is an audit failure
print("Client ready, model pinned to:", MODEL)

Client ready, model pinned to: gemini-3.5-flash-lite


## §2 · First call — define IBNR for a board member

In [5]:
response = client.models.generate_content(
    model=MODEL,
    contents="Define IBNR for a non-actuarial board member, in one line.",
)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(response.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
print("\n--- usage ---")
print(response.usage_metadata)   # token counts: you will care about these when agents multiply call volume

📋 GEMINI MODEL RESPONSE


IBNR (Incurred But Not Reported) represents the estimated financial reserve an organization must set aside to cover claims that have already occurred but have not yet been filed by the policyholder.


END OF MODEL RESPONSE

--- usage ---
cache_tokens_details=None cached_content_token_count=None candidates_token_count=38 candidates_tokens_details=None prompt_token_count=19 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=19
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=57 traffic_type=None


## §3 · CCCE — Clarity, Context, Constraints, Examples
The prompt below is the worked example from the slides: an IBNR commentary for ABC Health Q3 2024. Each bracketed fragment does exactly one job — edit any part without breaking the others.

**Exercise:** delete the Constraints block, re-run, and compare. Then rewrite the prompt for *your* line of business.

In [6]:
ccce_prompt = """
[Clarity] Write a two-paragraph commentary on the IBNR result for ABC Health Q3 2024.
[Context] Indemnity health book. Chain-ladder ultimate INR 186 Cr vs prior estimate INR 172 Cr.
Q3 saw a hospital network strike in two states.
[Constraints] Audience: appointed actuary peer-review meeting. Max 180 words.
Do not invent figures. Cite only the figures provided above.
[Example] Voice to match: "The Q2 ultimate of INR 164 Cr increased to INR 172 Cr after the network
expansion in Tier 2 cities..."
"""
resp = client.models.generate_content(model=MODEL, contents=ccce_prompt)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(resp.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

📋 GEMINI MODEL RESPONSE


The Q3 2024 valuation for the indemnity health book reflects an increase in the IBNR ultimate to INR 186 Cr, up from the prior estimate of INR 172 Cr. This upward development is primarily driven by the chain-ladder methodology, which captures the emerging claims volatility following the recent hospital network strikes across two key states.

The disruption in provider access significantly altered the reporting and settlement patterns for the period. We attribute the INR 14 Cr deterioration to the delayed submission of claims during the strike window and subsequent administrative backlogs, which have caused a deviation from historical completion factors. We are closely monitoring the post-strike stabilization to determine if this increase represents a one-time liquidity surge or a permanent shift in loss emergence trends for the portfolio.


END OF MODEL RESPONSE


### §3.1 · Demo 1 — the vague version, for contrast
Run the deliberately vague prompt below, then re-run the CCCE version above and **diff the outputs**. Same model, same cost — the entire quality delta is the prompt.

In [7]:
vague = "Write about IBNR for our board."
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(client.models.generate_content(model=MODEL, contents=vague).text[:800]))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
# Expect: a generic essay that INVENTS plausible numbers (we gave it none)
# and lands in a register somewhere between textbook and LinkedIn.

📋 GEMINI MODEL RESPONSE


This briefing note is designed for a Board of Directors. It balances technical accuracy with the strategic implications of IBNR (Incurred But Not Reported) reserves.

***

# Board Briefing: Understanding IBNR Reserves
**Subject:** Overview of IBNR, its financial impact, and governance considerations.

### 1. What is IBNR?
**IBNR** stands for **Incurred But Not Reported**. In the context of insurance and risk management, it represents the estimated liability for claims that have already occurred but have not yet been reported to the organization.

When a loss event happens (e.g., an accident, a medical diagnosis, or a cyber breach), there is often a time lag between the event and the moment the policyholder notifies the insurer. IBNR represents the "hidden" financial obligations currently s


END OF MODEL RESPONSE


### §3.2 · Demo 2 — one fact, two audiences
Audience is a prompt parameter. Same reserve-strengthening fact, rendered for a board member and for a new student. Note: the model *dresses* the fact we supply — it does not source it.

In [8]:
fact = ("We strengthened PMI hospitalisation reserves by INR 42 Cr "
        "following a sharp rise in empanelled-hospital tariffs.")

for audience, style in [
    ("board member", "2 sentences, business impact first, no jargon"),
    ("new actuarial student", "4 sentences, explain WHY tariffs drive PMI reserves, define terms"),
]:
    r = client.models.generate_content(
        model=MODEL,
        contents=f"Explain: {fact} For a {audience}. {style}")
    # Clear visual separation
    print("=" * 70)
    print("📋 GEMINI MODEL RESPONSE")
    print("=" * 70)

    display(Markdown(f"**{audience.upper()}**\n\n{r.text}"))

    print("\n" + "=" * 70)
    print("END OF MODEL RESPONSE")
    print("=" * 70)


📋 GEMINI MODEL RESPONSE


**BOARD MEMBER**

To protect the company’s bottom line, we increased hospitalisation reserves by INR 42 Cr to account for the recent, unexpected surge in medical service costs. This proactive adjustment ensures we maintain adequate financial buffers against higher-than-anticipated claims expenses, preserving our long-term solvency.


END OF MODEL RESPONSE
📋 GEMINI MODEL RESPONSE


**NEW ACTUARIAL STUDENT**

In Private Medical Insurance (PMI), "reserves" represent the money an insurer sets aside today to pay for future claims that have already been incurred but not yet settled. When empanelled hospitals—those with whom the insurer has a formal billing agreement—sharply increase their "tariffs" (the prices charged for surgeries, room rent, and medical procedures), the expected cost of those future claims rises significantly. Because the insurer must ensure they have enough capital to cover these inflated costs, they are legally and prudently required to increase their financial reserves by INR 42 Cr. Effectively, this reserve strengthening is an actuarial adjustment to account for the higher "loss severity" caused by medical inflation within the provider network.


END OF MODEL RESPONSE


### §3.3 · Demo 3 — few-shot examples tame formatting
Show, don't tell: two worked examples buy you the delimiter, the casing, the arrow convention, and no chatty preamble. **Exercise:** feed it a genuinely weird input and see whether the pattern holds.

In [9]:
prompt = """Convert each change to the format of the examples.

EXAMPLES
In: Hospitalisation frequency moved from 3.2% to 3.5% for ages 45+.
Out: HOSP_FREQ | age 45+ | 3.2% -> 3.5%
In: Claim severity trend up 40bps.
Out: SEV_TREND | all | +40bps

NOW CONVERT
In: CI incidence for cardiac conditions, ages 40-55, moves from 0.45% to 0.52%.
Out:"""
print(client.models.generate_content(model=MODEL, contents=prompt).text)


Out: CI_INC_CARDIAC | age 40-55 | 0.45% -> 0.52%


### §3.4 · Demo 4 — step-by-step reasoning (with a warning label)
Asking for steps improves reliability — it does **not** guarantee it. Re-run this cell three times: do the running totals stay identical? This is why the afternoon's agent does arithmetic in *Python* and lets Gemini narrate.

In [10]:
prompt = """A PMI policy has base premium INR 9,000 with relativities:
age band 46-55 = 1.45, sum insured 10L = 1.30, family floater = 1.10, NCB 30% = 0.70.
Walk through the premium calculation STEP BY STEP, showing the running
total after each factor, then state the final premium."""
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(client.models.generate_content(model=MODEL, contents=prompt).text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
# Check by hand: 9000 * 1.45 * 1.30 * 1.10 * 0.70 = 13,063.05


📋 GEMINI MODEL RESPONSE


To calculate the final premium for the Private Medical Insurance (PMI) policy, we apply the factors sequentially to the base premium. 

*Note: In insurance underwriting, factors are typically applied multiplicatively to the base premium.*

### **Step-by-Step Calculation:**

**Base Premium:** INR 9,000

**Step 1: Apply Age Band Factor (1.45)**
*   Calculation: $9,000 \times 1.45 = 13,050$
*   **Running Total: INR 13,050**

**Step 2: Apply Sum Insured Factor (1.30)**
*   Calculation: $13,050 \times 1.30 = 16,965$
*   **Running Total: INR 16,965**

**Step 3: Apply Family Floater Factor (1.10)**
*   Calculation: $16,965 \times 1.10 = 18,661.50$
*   **Running Total: INR 18,661.50**

**Step 4: Apply NCB Factor (0.70)**
*   Calculation: $18,661.50 \times 0.70 = 13,063.05$
*   **Running Total: INR 13,063.05**

***

### **Final Premium:**
The final premium, after applying all relativities, is **INR 13,063.05** (often rounded to **INR 13,063** depending on company policy).


END OF MODEL RESPONSE


### §3.5 · Demo review — the habit that IS the skill
1. CCCE moved quality more than a model upgrade would — specification beats horsepower.
2. Register control is leverage, but the model dresses facts; it doesn't source them.
3. Few-shot is a formatting contract — stress-test it before relying on it.
4. Step-by-step is transparency, not verified arithmetic.

**The loop:** prompt → output → review → edit prompt — the same loop you'll run on agent traces this afternoon.

## §4 · The hallucination demo — run it, believe it
We ask for a regulation that **does not exist**. The model will not say 'no such factor' — it will produce the most *plausible-sounding* answer, confidently.

⚠️ This exact failure mode reappears **inside your agent** in notebook 04 — and you will fix it with a guardrail tool.

In [11]:
hallucination_prompt = (
    "What is the IRDAI-mandated co-payment factor for senior-citizen PMI "
    "policies with sum insured above INR 10 lakh? "
    "Give the exact factor value and the section reference."
)
resp = client.models.generate_content(model=MODEL, contents=hallucination_prompt)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(resp.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
print("\n⚠️  Verify before you trust: there is no such published factor. "
      "Whatever appears above was constructed to be plausible, not true.")


📋 GEMINI MODEL RESPONSE


As of the latest regulatory updates from the Insurance Regulatory and Development Authority of India (IRDAI), there is **no specific, universal mandated co-payment factor** for senior citizen policies with a sum insured above INR 10 lakh.

### Clarification on the Regulatory Stance
In April 2024, the IRDAI issued the **Master Circular on Health Insurance (Ref: IRDAI/HLT/REG/CIR/001/2024)**, which overhauled several aspects of senior citizen coverage. 

1.  **Removal of Mandatory Co-payments:** The IRDAI has been moving toward the elimination of age-based mandatory co-payments. Under the new guidelines, insurers are encouraged to offer products without mandatory co-payments.
2.  **Product Design:** While the IRDAI mandates that insurers must offer health insurance to persons aged 65 and above, it does **not** set a fixed numeric "factor" for co-payments. Instead, the regulator allows insurers to decide the co-payment structure as part of the product filing (under the "Product Filing Guidelines"). 
3.  **The "No Rejection" Clause:** The critical mandate is that insurers can no longer refuse to cover senior citizens, and they must provide products that cater to this segment. If an insurer chooses to apply a co-payment to keep premiums affordable, it must be clearly disclosed in the **File and Use** process approved by the IRDAI.

### Section Reference
*   **Master Circular on Health Insurance (Dated: May 29, 2024):** Specifically **Section 5 (Provisions for Senior Citizens)**.
*   **Clause 5.1:** States that "Insurers shall ensure that they have health insurance products tailored to the needs of senior citizens."
*   **Clause 5.3:** Explicitly mentions that "Insurers shall not impose any mandatory co-payment on senior citizens which is not applicable to other policyholders," effectively barring discriminatory co-payments based purely on age.

### Important Note for Policyholders
If you are looking for a specific percentage (e.g., 10% or 20%), please be aware that **this is determined by the specific product's Terms and Conditions (T&C)** as filed by the insurer and approved by the IRDAI, rather than a universal statutory factor.

*   **If your policy has a co-payment:** It is a contractual agreement between you and the insurer.
*   **If you are shopping for a policy:** You can look for "Zero Co-payment" plans for senior citizens, which are now widely available following the IRDAI's directive to prioritize coverage for this demographic.

***Disclaimer:** I am an AI, not an insurance regulator. Regulatory circulars are subject to periodic amendments. You should verify the specific wording of your policy document (under the "Co-payment" clause) or consult the IRDAI’s [Bima Bharosa portal](https://bimabharosa.irdai.gov.in/) for the most current insurer-specific filings.*


END OF MODEL RESPONSE

⚠️  Verify before you trust: there is no such published factor. Whatever appears above was constructed to be plausible, not true.


## §5 · Structured output — because agents speak JSON
One config line guarantees parseable JSON. This is how every component of an agentic system exchanges data — prose is only for humans at the last step.

In [12]:
import json

prompt = """For individual PMI (health) cover, list 5 rating factors.
For each: name, direction (increase/decrease premium), one-line justification. Return JSON."""

resp = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"response_mime_type": "application/json"},
)
factors = json.loads(resp.text)   # guaranteed to parse
for f in factors:
    print(f)


{'name': 'Age', 'direction': 'Increase', 'justification': 'The probability and severity of health conditions rise significantly as an individual gets older.'}
{'name': 'Geographic Location', 'direction': 'Increase', 'justification': 'Costs for medical procedures and hospital facilities vary significantly by region and local cost of living.'}
{'name': 'Medical History', 'direction': 'Increase', 'justification': 'Pre-existing conditions or a history of chronic illness increase the likelihood of future claims.'}
{'name': 'Excess/Deductible Amount', 'direction': 'Decrease', 'justification': "A higher out-of-pocket contribution from the policyholder reduces the insurer's liability for small claims."}
{'name': 'Smoking Status', 'direction': 'Increase', 'justification': 'Tobacco use is strongly correlated with a higher risk of developing expensive-to-treat diseases like cancer and heart conditions.'}


## §6 · Review exercise — mark the model's homework
Treat the JSON above as a junior analyst's first draft and grade it:

1. Is every **direction** consistent with your priors?
2. Did it name factors your book doesn't collect (e.g. a wellness-programme discount, occupation class)?
3. What material factors are **missing** (BMI band? room-rent category?)
4. What would you still need before any of this goes near a tariff filing? *(Hint: magnitudes → a GLM run → notebook 02.)*

**The rule that survives today:** the reasoner narrates; tools know; humans sign.

---
**Log what you ran.** For anything regulatory: save the full prompt–response pair, the model id, and the timestamp — 'the AI wrote it' is not a defence without the receipt.

In [13]:
# Minimal call log — one CSV row per call. In production this is your observability stack.
import datetime, csv, pathlib

def log_call(prompt, response_text, model=MODEL, path="genai_call_log.csv"):
    new = not pathlib.Path(path).exists()
    with open(path, "a", newline="") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["ts_utc", "model", "prompt", "response"])
        w.writerow([datetime.datetime.now(datetime.UTC).isoformat(), model, prompt, response_text])

log_call(prompt, resp.text)
print("logged — this habit is checklist question 10 in miniature")

logged — this habit is checklist question 10 in miniature
